# 08. Confounding Check

This notebook is standalone and pulls required inputs from earlier notebooks.

It builds/loads `df_patient`, computes structural confounders per sample, and tests whether key metrics are associated with those confounders.


In [10]:
import os
from pathlib import Path

# Resolve project root robustly (works from notebooks/, project root, or Downloads/).
_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "Spatial HCC",
    Path.cwd().parent / "Spatial HCC",
]
for _root in _candidates:
    if (_root / "st_adata_scored.h5ad").exists() or (_root / "GSE238264_RAW").exists():
        os.chdir(_root)
        break

print("Working directory:", Path.cwd())


Working directory: /Users/prateek/Downloads/Spatial HCC


In [11]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import networkx as nx
import statsmodels.api as sm


def normalize_group_label(x):
    """Normalize response labels into canonical R / NR strings."""
    s = str(x).strip().upper()
    if s in {"R", "RESPONDER"}:
        return "R"
    if s in {"NR", "NONRESPONDER", "NON-RESPONDER", "NON_RESPONDER"}:
        return "NR"
    return s


In [12]:
# Load st_adata from memory or from notebook-02 output.
if "st_adata" not in globals():
    h5_path = Path("st_adata_scored.h5ad")
    if not h5_path.exists():
        raise FileNotFoundError("st_adata_scored.h5ad not found. Run notebook 02 first.")
    st_adata = sc.read_h5ad(h5_path.as_posix())

required_obs = ["sample", "spot_malignant"]
missing = [c for c in required_obs if c not in st_adata.obs.columns]
if missing:
    raise KeyError(f"st_adata.obs missing required columns: {missing}")

st_adata.obs["sample"] = st_adata.obs["sample"].astype(str)
st_by_sample = {
    s: st_adata[st_adata.obs["sample"] == s].copy()
    for s in sorted(st_adata.obs["sample"].unique())
}
print("Samples:", sorted(st_by_sample.keys()))


Samples: ['HCC1R', 'HCC2R', 'HCC3R', 'HCC4R', 'HCC5NR', 'HCC6NR', 'HCC7NR']


In [13]:
# Load df_patient from memory or notebook-06 / notebook-03 outputs.
required_df_patient_cols = ["sample", "group", "C_b_cyto", "C_malig_caf", "C_exh_cyto"]

df_patient_local = None
source_name = None

if "df_patient" in globals() and isinstance(df_patient, pd.DataFrame):
    if all(c in df_patient.columns for c in required_df_patient_cols):
        df_patient_local = df_patient.copy()
        source_name = "in-memory df_patient"

if df_patient_local is None:
    candidate_paths = [
        Path("sensitivity_outputs/df_patient_for_sici_sensitivity.csv"),
        Path("SICI_sensitivity_outputs/df_patient_for_sici_sensitivity.csv"),
        Path("supplement_exports/Table_S1_metrics_couplings_torus.csv"),
        Path("supplement_exports_split/Table_S1_master_metrics_baselines.csv"),
        Path("supplement_exports_v2/Table_S1_master_metrics_baselines_torus.csv"),
    ]
    for cp in candidate_paths:
        if cp.exists():
            tmp = pd.read_csv(cp)
            if all(c in tmp.columns for c in required_df_patient_cols):
                df_patient_local = tmp.copy()
                source_name = str(cp)
                break

if df_patient_local is None:
    raise FileNotFoundError(
        "Could not find a usable df_patient source. Run notebook 06 first (preferred) or notebook 03 exports."
    )

# Keep and standardize.
df_patient_local = df_patient_local[required_df_patient_cols + [c for c in ["SICI"] if c in df_patient_local.columns]].copy()
df_patient_local["sample"] = df_patient_local["sample"].astype(str)
df_patient_local["group"] = df_patient_local["group"].map(normalize_group_label)
df_patient_local = df_patient_local[df_patient_local["group"].isin(["R", "NR"])].copy()

# Deduplicate (e.g., if a table has multiple rows per sample from multiple scales).
if df_patient_local["sample"].duplicated().any():
    agg = {
        "group": "first",
        "C_b_cyto": "mean",
        "C_malig_caf": "mean",
        "C_exh_cyto": "mean",
    }
    if "SICI" in df_patient_local.columns:
        agg["SICI"] = "mean"
    df_patient_local = df_patient_local.groupby("sample", as_index=False).agg(agg)

# Compute default SICI if not present.
if "SICI" not in df_patient_local.columns:
    df_patient_local["SICI"] = (
        df_patient_local["C_b_cyto"]
        - df_patient_local["C_malig_caf"]
        - df_patient_local["C_exh_cyto"]
    )

print("df_patient source:", source_name)
print("df_patient shape:", df_patient_local.shape)
display(df_patient_local.sort_values("sample"))


df_patient source: sensitivity_outputs/df_patient_for_sici_sensitivity.csv
df_patient shape: (7, 6)


,sample,group,C_b_cyto,C_malig_caf,C_exh_cyto,SICI
0,HCC1R,R,0.017711,-0.033758,0.019049,0.032420
1,HCC2R,R,-0.032210,-0.057423,0.038262,-0.013049
2,HCC3R,R,0.007831,-0.040782,0.022126,0.026488
3,HCC4R,R,0.095418,-0.035658,0.061939,0.069138
4,HCC5NR,NR,-0.010910,-0.065038,-0.025713,0.079841
5,HCC6NR,NR,0.002226,-0.104665,0.032672,0.074218
6,HCC7NR,NR,0.032413,-0.070682,0.010254,0.092841


In [14]:
# Compute sample-level confounders from st_adata geometry/content.
# n_spots: number of spatial spots in sample
# tumor_frac: fraction of spots with spot_malignant > 0
# n_components: number of connected components in spatial graph

confounders = []
for sample, ad in st_by_sample.items():
    sq.gr.spatial_neighbors(ad, coord_type="grid")
    W = ad.obsp["spatial_connectivities"]

    n_spots = int(W.shape[0])
    f_malig = ad.obs["spot_malignant"].to_numpy(dtype=float)
    tumor_high = f_malig > 0.0
    tumor_frac = float(np.mean(tumor_high))

    G = nx.from_scipy_sparse_array(W)
    n_components = int(nx.number_connected_components(G))

    confounders.append(
        {
            "sample": sample,
            "n_spots": n_spots,
            "tumor_frac": tumor_frac,
            "n_components": n_components,
        }
    )

df_conf = pd.DataFrame(confounders).sort_values("sample").reset_index(drop=True)
display(df_conf)


INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           


,sample,n_spots,tumor_frac,n_components
0,HCC1R,3006,0.907186,2
1,HCC2R,2766,0.958424,3
2,HCC3R,2170,1.000000,5
3,HCC4R,3002,1.000000,11
4,HCC5NR,1320,0.990152,21
5,HCC6NR,2575,0.999612,7
6,HCC7NR,2453,1.000000,9


In [15]:
# Merge patient metrics with confounders.
df_all = df_patient_local.merge(df_conf, on="sample", how="inner")
if df_all.empty:
    raise ValueError("No overlapping samples between df_patient and confounders.")

display(df_all.sort_values("sample"))
print("Merged rows:", len(df_all))


,sample,group,C_b_cyto,C_malig_caf,C_exh_cyto,SICI,n_spots,tumor_frac,n_components
0,HCC1R,R,0.017711,-0.033758,0.019049,0.032420,3006,0.907186,2
1,HCC2R,R,-0.032210,-0.057423,0.038262,-0.013049,2766,0.958424,3
2,HCC3R,R,0.007831,-0.040782,0.022126,0.026488,2170,1.000000,5
3,HCC4R,R,0.095418,-0.035658,0.061939,0.069138,3002,1.000000,11
4,HCC5NR,NR,-0.010910,-0.065038,-0.025713,0.079841,1320,0.990152,21
5,HCC6NR,NR,0.002226,-0.104665,0.032672,0.074218,2575,0.999612,7
6,HCC7NR,NR,0.032413,-0.070682,0.010254,0.092841,2453,1.000000,9


Merged rows: 7


In [16]:
# Spearman correlations with potential confounders.
metrics = ["SICI", "C_exh_cyto", "C_malig_caf"]
conf_vars = ["n_spots", "tumor_frac", "n_components"]

rows = []
for m in metrics:
    for c in conf_vars:
        rho = df_all[m].corr(df_all[c], method="spearman")
        rows.append({"metric": m, "confounder": c, "spearman_rho": float(rho) if np.isfinite(rho) else np.nan})

corr_df = pd.DataFrame(rows)
display(corr_df)


,metric,confounder,spearman_rho
0,SICI,n_spots,-0.392857
1,SICI,tumor_frac,0.333562
2,SICI,n_components,0.714286
3,C_exh_cyto,n_spots,0.571429
4,C_exh_cyto,tumor_frac,0.148250
5,C_exh_cyto,n_components,-0.214286
6,C_malig_caf,n_spots,0.571429
7,C_malig_caf,tumor_frac,-0.222375
8,C_malig_caf,n_components,-0.392857


In [17]:
# Regression check: metric ~ group + n_spots + tumor_frac (+ optional n_components)
# Note: with small N (7), interpret p-values cautiously; effect direction/size is still informative.

def regression_test(metric, include_components=True):
    cols = ["group", "n_spots", "tumor_frac"] + (["n_components"] if include_components else [])
    X = df_all[cols].copy()
    X["group"] = (X["group"] == "R").astype(int)
    X = sm.add_constant(X, has_constant="add")
    y = df_all[metric].astype(float)

    model = sm.OLS(y, X).fit()
    return model

for metric in ["SICI", "C_exh_cyto", "C_malig_caf"]:
    model = regression_test(metric, include_components=True)
    print("\n===", metric, "===")
    print(model.summary())

# Standardized preprocessing block (as requested) for SICI model.
from sklearn.preprocessing import StandardScaler

X = df_all[["group", "n_spots", "tumor_frac", "n_components"]].copy()
X["group"] = (X["group"] == "R").astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = sm.add_constant(X_scaled)
model_scaled = sm.OLS(df_all["SICI"].astype(float), X_scaled).fit()

print("\n=== SICI with standardized predictors ===")
print(model_scaled.summary())



=== SICI ===
                            OLS Regression Results                            
Dep. Variable:                   SICI   R-squared:                       0.799
Model:                            OLS   Adj. R-squared:                  0.398
Method:                 Least Squares   F-statistic:                     1.992
Date:                Tue, 03 Mar 2026   Prob (F-statistic):              0.361
Time:                        20:06:59   Log-Likelihood:                 19.189
No. Observations:                   7   AIC:                            -28.38
Df Residuals:                       2   BIC:                            -28.65
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.2037      0.402

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 7 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 7 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 7 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with

In [18]:
# Save confounding-check outputs.
out_dir = Path("confounding_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

df_conf.to_csv(out_dir / "confounders_per_sample.csv", index=False)
df_all.to_csv(out_dir / "patient_metrics_with_confounders.csv", index=False)
corr_df.to_csv(out_dir / "spearman_metric_vs_confounder.csv", index=False)

print("Saved:")
print("-", out_dir / "confounders_per_sample.csv")
print("-", out_dir / "patient_metrics_with_confounders.csv")
print("-", out_dir / "spearman_metric_vs_confounder.csv")


Saved:
- confounding_outputs/confounders_per_sample.csv
- confounding_outputs/patient_metrics_with_confounders.csv
- confounding_outputs/spearman_metric_vs_confounder.csv
